# M3-B Research Runner — Colab Run All

Runs the fixed **SMA 5/20** research baseline end to end:
Spec → Governance → Data → Signal → Core `ExecutionEngine.run` → Evaluation → Registry → Report → Archive.

* This notebook only installs pinned code and calls `python -m strategy_agent.research`. It contains **no strategy or backtest logic**.
* The **synthetic** run uses the committed fixture and an *isolated* governance environment under `/content/m3b_env`. It never writes the repository's registry or holdout locks.
* The **real TSM** run is blocked until the M3-A market gate is GO. The `readiness` cell prints what is missing.
* Archive: `MyDrive/Quant_Research/strategy-research-agent/YYYYMMDD_HHMMSS_<commit>/` (Asia/Taipei). If Drive is not mounted, the run is kept locally and reported as `DRIVE_UNAVAILABLE`. It is never reported as backed up.
* No secrets are needed (both repositories are public). Do not paste tokens into this notebook.


In [ ]:
# --- configuration (pinned) ---
AGENT_REPO = "https://github.com/hh4832/strategy-research-agent.git"
AGENT_REF  = "feature/m3b-research-runner"   # replace with the merged commit SHA after review
CORE_SHA   = "2c6ba9ab7c336e7a4908744abef617e0dce713fd"  # BENCHMARK_PASS (M1 final)
RUN_ROOT   = "/content/m3b_env"              # isolated synthetic governance + registry
REPEAT     = 2                               # reruns of the same job (reproducibility check)


In [ ]:
# --- mount Google Drive (archive target). A failure here is reported later, never hidden ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Drive not mounted:", e)


In [ ]:
# --- install the agent at AGENT_REF and Core at CORE_SHA ---
import subprocess, sys
def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-4000:], r.stderr[-4000:]); r.check_returncode(); return r.stdout
sh("rm -rf /content/strategy-research-agent")
sh(f"git clone -q {AGENT_REPO} /content/strategy-research-agent")
sh(f"cd /content/strategy-research-agent && git checkout -q {AGENT_REF} && git rev-parse HEAD")
sh(f"cd /content/strategy-research-agent && pip install -q -e '.[test,research]' "
   f"'market-research-core @ git+https://github.com/hh4832/market-research-core@{CORE_SHA}'")


In [ ]:
# --- identity: agent commit, Core commit, benchmark status (derived from evidence, never set by hand) ---
sh("cd /content/strategy-research-agent && python -c \"from strategy_agent.adapters import mrc_adapter as A; "
   "from strategy_agent.research.pipeline import agent_identity; print(A.core_identity()); "
   "print(A.engine_info()); print(agent_identity())\"")


In [ ]:
# --- tests (unit + synthetic integration on the real Core) ---
sh("cd /content/strategy-research-agent && python -m pytest -q tests/test_m3b_unit.py tests/test_m3b_integration.py "
   "tests/test_m1_core_compat.py")


In [ ]:
# --- M3-B2: real-market readiness (expected: NOT_READY until Codex M3-A is GO) ---
r = subprocess.run("cd /content/strategy-research-agent && python -m strategy_agent.research readiness --market IBKR_TSM",
                   shell=True, text=True, capture_output=True)
print(r.stdout); print("exit", r.returncode, "(3 = gate closed: the real TSM run is NOT started)")


In [ ]:
# --- M3-B1: synthetic run(s), archived to Drive, reproducibility check ---
r = subprocess.run(f"cd /content/strategy-research-agent && python -m strategy_agent.research synthetic "
                   f"--run-root {RUN_ROOT} --drive --repeat {REPEAT}", shell=True, text=True, capture_output=True)
print(r.stdout[-12000:]); print(r.stderr[-4000:]); print("exit", r.returncode)
SYNTH_EXIT = r.returncode


In [ ]:
# --- optional: pre-registered cost sensitivity (0x, 1x, 2x, 4x), same parameters -> one family trial ---
r = subprocess.run(f"cd /content/strategy-research-agent && python -m strategy_agent.research synthetic "
                   f"--run-root {RUN_ROOT} --drive --cost-sensitivity", shell=True, text=True, capture_output=True)
print(r.stdout[-8000:]); print("exit", r.returncode)


In [ ]:
# --- show the newest archived report ---
import pathlib, json
from IPython.display import Markdown, display
root = pathlib.Path('/content/drive/MyDrive/Quant_Research/strategy-research-agent')
runs = sorted(p for p in root.glob('2*') if (p / 'manifest.json').exists()) if root.exists() else []
print("archived runs:", [p.name for p in runs][-10:])
if runs:
    print((runs[-1] / 'run_info.txt').read_text())
    display(Markdown((runs[-1] / 'research_report.md').read_text()))


In [ ]:
# --- final check: fail loudly if the synthetic run, reproducibility or the Drive archive did not succeed ---
assert SYNTH_EXIT == 0, "synthetic run failed, was not reproducible, or Drive archive unavailable (see output above)"
for p in runs[-REPEAT:]:
    man = json.loads((p / 'manifest.json').read_text())['files']
    import hashlib
    for name, v in man.items():
        assert hashlib.sha256((p / name).read_bytes()).hexdigest() == v['sha256'], (p, name)
print("M3-B1 synthetic MVP: OK; archive verified:", [p.name for p in runs[-REPEAT:]])
